In [1]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

rd.open_session()

<refinitiv.data.session.Definition object at 0x119f2e030 {name='workspace'}>

In [2]:
universe = ['NESTE.HE']

fields = [
    "TR.PriceClose.date",
    "TR.PriceClose", # "dirty price" without dividends
]

params = {
    "SDate": "2024-01-01",
    "EDate": "2026-03-05",
    "Frq": "D",
    "Curn": "EUR"
}

df = rd.get_data(universe=universe, 
                 fields=fields, 
                 parameters=params)


df.head()


,Instrument,Date,Price Close
0,NESTE.HE,2024-01-02,32.48
1,NESTE.HE,2024-01-03,31.8
2,NESTE.HE,2024-01-04,32.27
3,NESTE.HE,2024-01-05,32.4
4,NESTE.HE,2024-01-08,32.27


#### Aritmeettinen muutos

In [3]:
df['Daily_Return'] = df['Price Close'].pct_change()

#### Vol30D

In [5]:
df['Vol_30d'] = df['Daily_Return'].rolling(30).std() * np.sqrt(252)

#### RSI30D

In [7]:
delta = df['Price Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(30).mean()
avg_loss = loss.rolling(30).mean()

rs = avg_gain / avg_loss
df['RSI_30d'] = 100 - (100 / (1 + rs))

In [8]:
df

,Instrument,Date,Price Close,Daily_Return,Vol_30d,RSI_30d
0,NESTE.HE,2024-01-02,32.48,<NA>,NaN,NaN
1,NESTE.HE,2024-01-03,31.8,-0.020936,NaN,NaN
2,NESTE.HE,2024-01-04,32.27,0.01478,NaN,NaN
3,NESTE.HE,2024-01-05,32.4,0.004029,NaN,NaN
4,NESTE.HE,2024-01-08,32.27,-0.004012,NaN,NaN
...,...,...,...,...,...,...
540,NESTE.HE,2026-02-27,21.18,-0.005634,0.328720,54.909820
541,NESTE.HE,2026-03-02,22.65,0.069405,0.372963,63.470320
542,NESTE.HE,2026-03-03,22.71,0.002649,0.372960,63.201472
543,NESTE.HE,2026-03-04,22.53,-0.007926,0.369014,60.303894
